In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("My Uber Drives-2016.csv")
df.head()


In [ ]:
print("Dataset Shape:", df.shape)
print("\nDataset Information:")
df.info()

print("\nSummary Statistics:")
display(df.describe(include="all"))


In [ ]:

df.columns = [
    "Start_Date",
    "End_Date",
    "Category",
    "Start",
    "Stop",
    "Miles",
    "Purpose"
]

df.head()


In [ ]:

print(df.isnull().sum())


In [ ]:

df["Start_Date"] = pd.to_datetime(df["Start_Date"], errors="coerce")
df["End_Date"] = pd.to_datetime(df["End_Date"], errors="coerce")

# Fill missing Purpose
df["Purpose"] = df["Purpose"].fillna("Unknown")

# Remove rows missing essential fields
df.dropna(subset=["Start_Date", "End_Date", "Category", "Start", "Stop", "Miles"], inplace=True)

print("Shape after cleaning:", df.shape)
print(df.isnull().sum())


In [ ]:

df["Hour"] = df["Start_Date"].dt.hour
df["Day"] = df["Start_Date"].dt.day
df["Month"] = df["Start_Date"].dt.month
df["Day_of_Week"] = df["Start_Date"].dt.day_name()
df["Duration_Minutes"] = (
    df["End_Date"] - df["Start_Date"]
).dt.total_seconds() / 60

df["Route"] = df["Start"].astype(str) + " → " + df["Stop"].astype(str)

df.head()


In [ ]:

print(df["Category"].value_counts())

plt.figure(figsize=(7,5))
sns.countplot(data=df, x="Category")
plt.title("Business vs Personal Trips")
plt.show()


In [ ]:

print(df["Purpose"].value_counts())

plt.figure(figsize=(10,5))
sns.countplot(
    data=df,
    y="Purpose",
    order=df["Purpose"].value_counts().index
)
plt.title("Trips by Purpose")
plt.show()


In [ ]:

hourly_trips = df["Hour"].value_counts().sort_index()

plt.figure(figsize=(10,5))
plt.plot(hourly_trips.index, hourly_trips.values, marker="o")
plt.xlabel("Hour")
plt.ylabel("Number of Trips")
plt.title("Trips by Hour")
plt.grid(True)
plt.show()


In [ ]:

day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

day_trips = df["Day_of_Week"].value_counts().reindex(day_order)

plt.figure(figsize=(9,5))
day_trips.plot(kind="bar")
plt.title("Trips by Day of Week")
plt.xlabel("Day")
plt.ylabel("Number of Trips")
plt.xticks(rotation=45)
plt.show()


In [ ]:

monthly_trips = df["Month"].value_counts().sort_index()

plt.figure(figsize=(9,5))
monthly_trips.plot(kind="bar")
plt.title("Monthly Uber Trips")
plt.xlabel("Month")
plt.ylabel("Number of Trips")
plt.show()


In [ ]:

print(df["Miles"].describe())

plt.figure(figsize=(9,5))
sns.histplot(df["Miles"], bins=30, kde=True)
plt.title("Distribution of Trip Distance")
plt.xlabel("Miles")
plt.show()


In [ ]:

top_start = df["Start"].value_counts().head(10)
print(top_start)

plt.figure(figsize=(10,5))
top_start.plot(kind="bar")
plt.title("Top 10 Starting Locations")
plt.xlabel("Location")
plt.ylabel("Number of Trips")
plt.xticks(rotation=45)
plt.show()


In [ ]:

top_stop = df["Stop"].value_counts().head(10)
print(top_stop)

plt.figure(figsize=(10,5))
top_stop.plot(kind="bar")
plt.title("Top 10 Destination Locations")
plt.xlabel("Location")
plt.ylabel("Number of Trips")
plt.xticks(rotation=45)
plt.show()


In [ ]:

top_routes = df["Route"].value_counts().head(10)
print(top_routes)

plt.figure(figsize=(10,5))
top_routes.sort_values().plot(kind="barh")
plt.title("Top 10 Most Common Routes")
plt.xlabel("Number of Trips")
plt.show()


In [ ]:

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Category", y="Miles")
plt.title("Trip Distance by Category")
plt.show()


In [ ]:

purpose_distance = df.groupby("Purpose")["Miles"].mean().sort_values()
print(purpose_distance)

plt.figure(figsize=(9,5))
purpose_distance.plot(kind="barh")
plt.title("Average Distance by Trip Purpose")
plt.xlabel("Average Miles")
plt.show()


In [ ]:

numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(9,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Select features
X = df[[
    "Miles",
    "Hour",
    "Day",
    "Month",
    "Duration_Minutes"
]].copy()

y = df["Category"].copy()

# Remove invalid infinite values
X = X.replace([np.inf, -np.inf], np.nan)
valid = X.notna().all(axis=1) & y.notna()

X = X.loc[valid]
y = y.loc[valid]

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


In [ ]:

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

plt.figure(figsize=(8,5))
importance.sort_values().plot(kind="barh")
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.show()
